In [1]:
import torch
import torch_geometric

print("Torch:", torch.__version__)
print("PyG:", torch_geometric.__version__)

Torch: 2.11.0+cpu
PyG: 2.7.0


In [2]:
from torch_geometric.nn.models.tgn import (
    TGNMemory,
    IdentityMessage,
    LastAggregator,
    LastNeighborLoader
)

print("TGN modules imported successfully")

TGN modules imported successfully


In [3]:
import numpy as np
import torch

data = np.load(
    "nft_graph_dataset.npz",
    allow_pickle=True
)

src = torch.tensor(data["src"], dtype=torch.long)
dst = torch.tensor(data["dst"], dtype=torch.long)

timestamps = torch.tensor(
    data["timestamps"],
    dtype=torch.long
)

edge_feat = torch.tensor(
    data["edge_feat"],
    dtype=torch.float32
)

labels = torch.tensor(
    data["labels"],
    dtype=torch.float32
)

train_mask = torch.tensor(data["train_mask"])
val_mask   = torch.tensor(data["val_mask"])
test_mask  = torch.tensor(data["test_mask"])

num_nodes = int(
    max(src.max(), dst.max())
) + 1

print("Nodes:", num_nodes)
print("Edges:", len(src))
print("Edge feature dim:", edge_feat.shape[1])

Nodes: 333077
Edges: 2713386
Edge feature dim: 9


In [5]:
print(
    "Already sorted:",
    bool(torch.all(
        timestamps[:-1] <= timestamps[1:]
    ))
)

Already sorted: True


In [6]:
train_idx = torch.where(train_mask)[0]
val_idx   = torch.where(val_mask)[0]
test_idx  = torch.where(test_mask)[0]

print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

print()

print(
    "Train Fraud:",
    int(labels[train_idx].sum())
)

print(
    "Val Fraud:",
    int(labels[val_idx].sum())
)

print(
    "Test Fraud:",
    int(labels[test_idx].sum())
)

Train: 1899370
Val: 407008
Test: 407008

Train Fraud: 15818
Val Fraud: 1684
Test Fraud: 1451


In [7]:
from torch_geometric.nn.models.tgn import (
    TGNMemory,
    IdentityMessage,
    LastAggregator
)

memory_dim = 100
time_dim = 100

memory = TGNMemory(
    num_nodes=num_nodes,
    raw_msg_dim=edge_feat.shape[1],
    memory_dim=memory_dim,
    time_dim=time_dim,
    message_module=IdentityMessage(
        edge_feat.shape[1],
        memory_dim,
        time_dim
    ),
    aggregator_module=LastAggregator()
)

In [8]:
import torch.nn as nn

class EdgePredictor(nn.Module):

    def __init__(self,
                 memory_dim,
                 edge_feat_dim):

        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(
                memory_dim * 2 +
                edge_feat_dim,
                256
            ),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                256,
                128
            ),
            nn.ReLU(),

            nn.Linear(
                128,
                1
            )
        )

    def forward(
        self,
        z_src,
        z_dst,
        edge_feat
    ):

        x = torch.cat(
            [
                z_src,
                z_dst,
                edge_feat
            ],
            dim=1
        )

        return self.mlp(x).squeeze(-1)

In [9]:
predictor = EdgePredictor(
    memory_dim=memory_dim,
    edge_feat_dim=edge_feat.shape[1]
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

memory = memory.to(DEVICE)
predictor = predictor.to(DEVICE)

print(DEVICE)

cpu


In [11]:
from torch.utils.data import DataLoader

BATCH_SIZE = 5000

train_loader = DataLoader(
    train_idx.numpy(),
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_idx.numpy(),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_idx.numpy(),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 380
Val batches: 82
Test batches: 82


In [12]:
import torch.optim as optim

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    list(memory.parameters()) +
    list(predictor.parameters()),
    lr=1e-3
)

In [13]:
def train_one_epoch():

    memory.train()
    predictor.train()

    memory.reset_state()

    total_loss = 0

    for batch_idx in train_loader:

        batch_idx = torch.tensor(
            batch_idx,
            dtype=torch.long
        )

        s = src[batch_idx].to(DEVICE)
        d = dst[batch_idx].to(DEVICE)

        t = timestamps[batch_idx].to(DEVICE)

        msg = edge_feat[batch_idx].to(DEVICE)

        y = labels[batch_idx].to(DEVICE)

        optimizer.zero_grad()

        z, last_update = memory(
            torch.cat([s, d]).unique()
        )

        z_src = z[
            torch.searchsorted(
                torch.cat([s, d]).unique(),
                s
            )
        ]

        z_dst = z[
            torch.searchsorted(
                torch.cat([s, d]).unique(),
                d
            )
        ]

        logits = predictor(
            z_src,
            z_dst,
            msg
        )

        loss = criterion(
            logits,
            y
        )

        loss.backward()

        optimizer.step()

        memory.update_state(
            s,
            d,
            t,
            msg
        )

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [14]:
help(memory.forward)

Help on method forward in module torch_geometric.nn.models.tgn:

forward(n_id: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor] method of torch_geometric.nn.models.tgn.TGNMemory instance
    Returns, for all nodes :obj:`n_id`, their current memory and their
    last updated timestamp.



In [15]:
from torch_geometric.nn.models.tgn import *

dir()

['BATCH_SIZE',
 'Callable',
 'DEVICE',
 'DataLoader',
 'Dict',
 'EdgePredictor',
 'GRUCell',
 'IdentityMessage',
 'In',
 'LastAggregator',
 'LastNeighborLoader',
 'Linear',
 'MeanAggregator',
 'Out',
 'TGNMemory',
 'TGNMessageStoreType',
 'Tensor',
 'TimeEncoder',
 'Tuple',
 '_',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'copy',
 'criterion',
 'data',
 'dst',
 'edge_feat',
 'exit',
 'get_ipython',
 'labels',
 'memory',
 'memory_dim',
 'nn',
 'np',
 'num_nodes',
 'open',
 'optim',
 'optimizer',
 'predictor',
 'quit',
 'scatter',
 'scatter_argmax',
 'src',
 'test_idx',
 'test_loader',
 'test_mask',
 'time_dim',
 'timestamps',
 'torch',
 'torch_geometric',
 'train_idx',
 'train_loader',
 'train_mask',
 'train_one_epoch',

In [16]:
from torch_geometric.nn.models import tgn

print(dir(tgn))

['Callable', 'Dict', 'GRUCell', 'IdentityMessage', 'LastAggregator', 'LastNeighborLoader', 'Linear', 'MeanAggregator', 'TGNMemory', 'TGNMessageStoreType', 'Tensor', 'TimeEncoder', 'Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'copy', 'scatter', 'scatter_argmax', 'torch', 'zeros']
